# Final model interpretation

## Imports

In [ ]:
#Install requirements
%pip install -r "../Requirements.txt"

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import pandas as pd
import numpy as np
import sys
import warnings

import shap
from collections import Counter
import matplotlib.pyplot as plt

project_dir = Path.cwd().parent

sys.path.insert(0, str(Path.cwd().parent)) # import src/
warnings.filterwarnings("ignore")

pd.set_option("display.width", 220, "display.max_columns", 30)

from src import build_xy, load_anndata, load_per_family, load_model

HVG_DATA_DIR = project_dir / 'data' / 'HVGs'
FIG_DIR = project_dir / 'figures' / 'shap'


## Dataset

In [ ]:
FAMILIES = {
    "KenyonCells":  (HVG_DATA_DIR / "Kenyon_cells_train_hvg.h5ad",  HVG_DATA_DIR / "Kenyon_cells_test_hvg.h5ad"),
    "OpticLobe":    (HVG_DATA_DIR / "Optic_lobe_train_hvg.h5ad",    HVG_DATA_DIR / "Optic_lobe_test_hvg.h5ad"),
    "Monoaminergic":(HVG_DATA_DIR / "Monoaminergic_train_hvg.h5ad",HVG_DATA_DIR / "Monoaminergic_test_hvg.h5ad"),
    "Glia":         (HVG_DATA_DIR / "Glia_train_hvg.h5ad",         HVG_DATA_DIR / "Glia_test_hvg.h5ad"),
    "Peptidergic":  (HVG_DATA_DIR / "Peptidergic_train_hvg.h5ad",  HVG_DATA_DIR / "Peptidergic_test_hvg.h5ad"),
    "Clock":        (HVG_DATA_DIR / "Clock_train_hvg.h5ad",        HVG_DATA_DIR / "Clock_test_hvg.h5ad"),
}
FAMILIES = {f: p for f, p in FAMILIES.items() if p[0].exists() and p[1].exists()}

## Final model load and fit

In [3]:
final_model_per_family = load_per_family("../results/final_model_per_family",
                                         name="final_model_per_family")

In [4]:
# Predict on test sets
test_scores = {}
model_per_family = {}
for fam, (train, test) in FAMILIES.items():
    Xte, yte, _ = build_xy(load_anndata(test), extra_obs=("nUMI",))
    pipe = load_model(f"../results/final_model_per_family/{fam}/final_XGBoost.joblib")
    pred = pipe.predict(Xte)
    model_per_family[fam] = pipe
    test_scores[fam] = {"MAE": float(np.mean(np.abs(yte - pred))),
                        "RMSE": float(np.sqrt(np.mean((yte - pred) ** 2)))}
    
test_scores_df = pd.DataFrame.from_dict(test_scores, orient='index')
display(test_scores_df)

,MAE,RMSE
KenyonCells,2.523939,4.501975
OpticLobe,2.684812,4.980188
Monoaminergic,4.201943,6.483697
Glia,5.552284,7.623145
Peptidergic,4.226700,6.662716
Clock,4.833010,7.639812


## SHAP values per family

In [5]:
FIG_DIR.mkdir(parents=True, exist_ok=True)

shap_per_family = {}

for fam, (train, _test) in FAMILIES.items():
    X, _, feat = build_xy(load_anndata(train), extra_obs=("nUMI",))
    pipe = model_per_family[fam]
    selector = pipe.named_steps['selector']
    model = pipe.named_steps['model']

    X_sel = selector.transform(X)
    genes = list(selector.get_feature_names_out(feat))

    explainer = shap.TreeExplainer(model)
    shap_vals = explainer.shap_values(X_sel)

    shap_per_family[fam] = {"values": shap_vals, "X": X_sel, "genes": genes}

    plt.figure()
    shap.summary_plot(shap_vals, X_sel, feature_names=genes, show=False, max_display=20)
    plt.title(f"SHAP summary plot for {fam} on Age (days)")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"shap_summary_{fam}.png", dpi=150, bbox_inches='tight')
    plt.close()
    print(f"{fam}: {X_sel.shape[0]} cells x {X_sel.shape[1]} genes")
    

KenyonCells: 2278 cells x 100 genes
OpticLobe: 6365 cells x 100 genes
Monoaminergic: 748 cells x 100 genes
Glia: 2991 cells x 100 genes
Peptidergic: 907 cells x 100 genes
Clock: 412 cells x 100 genes


## Per family global importance

In [6]:
importance = {fam: pd.Series(np.abs(d["values"]).mean(axis=0), index=d["genes"])
              .sort_values(ascending=False)
              for fam, d in shap_per_family.items()}

# in case we want to exclude library size
# importance = {fam: pd.Series(np.abs(d["values"]).mean(axis=0), index=d["genes"])
#               .drop("nUMI", errors="ignore").sort_values(ascending=False)
#               for fam, d in shap_per_family.items()}

top_genes_table = pd.DataFrame({fam: s.head(15).index for fam, s in importance.items()})
display(top_genes_table)

,KenyonCells,OpticLobe,Monoaminergic,Glia,Peptidergic,Clock
0,nUMI,mt:lrRNA,mt:lrRNA,mt:lrRNA,mt:lrRNA,mt:lrRNA
1,noe,sta,sta,nUMI,sta,RpS14b
2,mt:lrRNA,TM4SF,MRE16,MRE16,TM4SF,nUMI
3,TM4SF,MRE16,TM4SF,Tsp42Ed,MRE16,Hsromega
4,Hsromega,Hsromega,Hsromega,GstD1,Hsromega,TM4SF
5,mt:CoIII,nUMI,mt:CoIII,CG10433,mt:CoIII,sta
6,CG10077,mt:CoIII,nUMI,CG9377,Gapdh1,mt:CoIII
7,Ddc,Atox1,RpL22,CR43651,RpL3,cathD
8,CG34449,snRNA:7SK,Cyt-c-p,Tsf1,Cyt-c-p,CG9377
9,FoxP,Msp300,RpL39,Cyt-c-p,HmgD,Gapdh1


## Cross family importance

In [11]:
TOP_N = 30
top_sets_30 = {fam: set(s.head(TOP_N).index) for fam, s in importance.items()}
fams = list(top_sets_30)

jaccard_matrix_30 = pd.DataFrame(index=fams, columns=fams, dtype=float)
for a in fams:
    for b in fams:
        u, i = top_sets_30[a] | top_sets_30[b], top_sets_30[a] & top_sets_30[b]
        jaccard_matrix_30.loc[a, b] = len(i) / len(u) if u else 0.0
display(jaccard_matrix_30.round(2))

shared_genes = pd.Series(Counter(g for s in top_sets_30.values() for g in s)).sort_values(ascending=False)
display(shared_genes.head(15))

,KenyonCells,OpticLobe,Monoaminergic,Glia,Peptidergic,Clock
KenyonCells,1.00,0.13,0.13,0.05,0.07,0.09
OpticLobe,0.13,1.00,0.20,0.11,0.15,0.22
Monoaminergic,0.13,0.20,1.00,0.09,0.30,0.11
Glia,0.05,0.11,0.09,1.00,0.07,0.13
Peptidergic,0.07,0.15,0.30,0.07,1.00,0.20
Clock,0.09,0.22,0.11,0.13,0.20,1.00


mt:lrRNA    6
Hsromega    6
nUMI        5
mt:CoIII    5
TM4SF       5
sta         4
MRE16       4
CG31808     3
HmgD        3
Dscam4      3
Gapdh1      3
Xrp1        3
Cyt-c-p     3
Msp300      2
chrb        2
dtype: int64

In [12]:
TOP_N = 20
top_sets_20 = {fam: set(s.head(TOP_N).index) for fam, s in importance.items()}
fams = list(top_sets_20)

jaccard_matrix_20 = pd.DataFrame(index=fams, columns=fams, dtype=float)
for a in fams:
    for b in fams:
        u, i = top_sets_20[a] | top_sets_20[b], top_sets_20[a] & top_sets_20[b]
        jaccard_matrix_20.loc[a, b] = len(i) / len(u) if u else 0.0
display(jaccard_matrix_20.round(2))

,KenyonCells,OpticLobe,Monoaminergic,Glia,Peptidergic,Clock
KenyonCells,1.00,0.14,0.18,0.08,0.11,0.14
OpticLobe,0.14,1.00,0.29,0.14,0.25,0.25
Monoaminergic,0.18,0.29,1.00,0.14,0.29,0.18
Glia,0.08,0.14,0.14,1.00,0.11,0.14
Peptidergic,0.11,0.25,0.29,0.11,1.00,0.25
Clock,0.14,0.25,0.18,0.14,0.25,1.00


In [13]:
TOP_N = 10
top_sets_10 = {fam: set(s.head(TOP_N).index) for fam, s in importance.items()}
fams = list(top_sets_10)

jaccard_matrix_10 = pd.DataFrame(index=fams, columns=fams, dtype=float)
for a in fams:
    for b in fams:
        u, i = top_sets_10[a] | top_sets_10[b], top_sets_10[a] & top_sets_10[b]
        jaccard_matrix_10.loc[a, b] = len(i) / len(u) if u else 0.0
display(jaccard_matrix_10.round(2))

,KenyonCells,OpticLobe,Monoaminergic,Glia,Peptidergic,Clock
KenyonCells,1.00,0.33,0.33,0.11,0.25,0.33
OpticLobe,0.33,1.00,0.54,0.18,0.43,0.43
Monoaminergic,0.33,0.54,1.00,0.25,0.54,0.43
Glia,0.11,0.18,0.25,1.00,0.18,0.18
Peptidergic,0.25,0.43,0.54,0.18,1.00,0.43
Clock,0.33,0.43,0.43,0.18,0.43,1.00


In [14]:
def plot_jaccard_heatmap(jaccard_matrix, path, title=None, cmap="viridis"):
    fams = list(jaccard_matrix.index)
    M = jaccard_matrix.astype(float).values
    n = len(fams)

    fig, ax = plt.subplots(figsize=(1.1 * n + 2, 1.1 * n + 1))
    im = ax.imshow(M, cmap=cmap, vmin=0, vmax=1)

    for i in range(n):
        for j in range(n):
            val = M[i, j]
            ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=8,
                    color="white" if val < 0.5 else "black")

    ax.set_xticks(range(n)); ax.set_xticklabels(fams, rotation=40, ha="right", fontsize=9)
    ax.set_yticks(range(n)); ax.set_yticklabels(fams, fontsize=9)
    ax.set_title(title or "Top-gene overlap across families (Jaccard)", fontsize=13)

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Jaccard index")

    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"Saved heatmap to {path}")


In [15]:
plot_jaccard_heatmap(jaccard_matrix=jaccard_matrix_10, path=FIG_DIR / "jaccard_topgenes_10.png")
plot_jaccard_heatmap(jaccard_matrix=jaccard_matrix_20, path=FIG_DIR / "jaccard_topgenes_20.png")
plot_jaccard_heatmap(jaccard_matrix=jaccard_matrix_30, path=FIG_DIR / "jaccard_topgenes_30.png")

Saved heatmap to /home/giorgos/Documents/MLCB/MLCB_project/figures/shap/jaccard_topgenes_10.png
Saved heatmap to /home/giorgos/Documents/MLCB/MLCB_project/figures/shap/jaccard_topgenes_20.png
Saved heatmap to /home/giorgos/Documents/MLCB/MLCB_project/figures/shap/jaccard_topgenes_30.png
